<a href="https://colab.research.google.com/github/Sayak-coder/HateSpeech-detection/blob/main/Subtask_A_%5BIT%5D_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers

In [2]:
import pandas as pd
import numpy as np
import torch

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
df_IT = pd.read_csv('/content/StereoQueerEval_IT_training.tsv', sep='\t')
df_IT.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_IT_0001,"Nivan transgender a 17 anni: ""Il mio percorso ...",Nivan è un ragazzo transgender Ftm di soli 17 ...,Sono transgender ma ho paura di dirlo ai miei ...,no,no,none
1,training_IT_0002,"Casillas annuncia di essere gay, poi parla di ...","Travolto dalle polemiche, Casillas è stato cos...",E cosa sarà mai guardate cosa succede nel mond...,no,yes_implicit,group_g
2,training_IT_0003,Schlein: «Con governo Meloni l’Italia è scivol...,«Sono sempre felice quando c’è il riconoscimen...,Ma fate silenzio il gay pride e finito ora and...,yes,yes_explicit,group_lgbtqia+
3,training_IT_0004,"Olimpiadi Parigi 2024, la pugile Imane Khelif ...",Dopo che Angela Carini alle Olimpiadi di Parig...,Donna Trans NON ESISTE! Un uomo non puo` diven...,yes,yes_explicit,group_t
4,training_IT_0005,"Roma Pride 2018, l'inno ""Ma che ti importa"": ""...",Sabato 9 giugno il Roma Pride scende in piazza...,Allora che vi importa se uno dice che prenderl...,no,yes_explicit,"group_g,lgbtqia+"


In [5]:
df = df_IT

In [6]:
df.isnull().sum()

,0
StereoQueerEval_id,0
yt_title,0
yt_description,0
yt_comment,0
stereotype,0
hate_speech,0
target,0


In [7]:
df['stereotype'].value_counts()

,count
stereotype,
no,1752
yes,648


#Experiment 2.A
only comment as input

In [ ]:
X = list(df['yt_comment'])
y = df['stereotype'].map({'no': 0, 'yes': 1}).to_list()

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=45)

In [ ]:
def build_inputs(d):
    comments = list(d['yt_comment'])
    return comments

def encode(d, tokenizer, max_length=256):
    comments = build_inputs(d)
    return tokenizer(comments, truncation=True, max_length=max_length)

In [ ]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###distilbert-multilingual

In [ ]:
from transformers import AutoTokenizer

MODEL = f"distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [ ]:
train_encodings = tokenizer(X_train, truncation=True, max_length=256)
val_encodings = tokenizer(X_val, truncation=True, max_length=256)
test_encodings  = tokenizer(X_test,  truncation=True, max_length=256)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset  = TextDataset(test_encodings,  y_test)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [ ]:
from transformers import DistilBertForSequenceClassification

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,0.752661
20,0.726162
30,0.695159
40,0.631178
50,0.653572
60,0.582585
70,0.656210
80,0.588651
90,0.584936
100,0.589384


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1200, training_loss=0.37443374400337537, metrics={'train_runtime': 255.966, 'train_samples_per_second': 37.505, 'train_steps_per_second': 4.688, 'total_flos': 288822394935264.0, 'train_loss': 0.37443374400337537, 'epoch': 5.0})

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.811     0.833     0.822       180
         yes      0.455     0.417     0.435        60

    accuracy                          0.729       240
   macro avg      0.633     0.625     0.628       240
weighted avg      0.722     0.729     0.725       240

[[150  30]
 [ 35  25]]


###twitter-roberta-base-hate-latest

In [ ]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [ ]:
train_encodings = tokenizer(X_train, truncation=True, max_length=256)
val_encodings = tokenizer(X_val, truncation=True, max_length=256)
test_encodings  = tokenizer(X_test,  truncation=True, max_length=256)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset  = TextDataset(test_encodings,  y_test)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [ ]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.295962
20,0.744157
30,0.737463
40,0.626002
50,0.585795
60,0.556790
70,0.611913
80,0.659265
90,0.645754
100,0.568677


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.47175260543823244, metrics={'train_runtime': 497.1562, 'train_samples_per_second': 19.31, 'train_steps_per_second': 1.207, 'total_flos': 921497138075520.0, 'train_loss': 0.47175260543823244, 'epoch': 5.0})

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.787     0.861     0.822       180
         yes      0.419     0.300     0.350        60

    accuracy                          0.721       240
   macro avg      0.603     0.581     0.586       240
weighted avg      0.695     0.721     0.704       240

[[155  25]
 [ 42  18]]


#Experiment 2.B
comment + title as input

In [8]:
df.columns

Index(['StereoQueerEval_id', 'yt_title', 'yt_description', 'yt_comment',
       'stereotype', 'hate_speech', 'target'],
      dtype='object')

In [27]:
X = (df.drop(['StereoQueerEval_id', 'yt_description',
       'stereotype', 'hate_speech', 'target'], axis =1))
y = df['stereotype'].map({'no': 0, 'yes': 1}).to_list()

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 45)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 45)

In [29]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    ctx = title
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [30]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###twitter-roberta-base-hate-latest

In [31]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [32]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset  = TextDataset(test_encodings,  y_test)

In [33]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [34]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [35]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.272932
20,0.708689
30,0.720267
40,0.615729
50,0.606441
60,0.567687
70,0.620029
80,0.620546
90,0.612199
100,0.512137


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.47776405652364096, metrics={'train_runtime': 377.6264, 'train_samples_per_second': 25.422, 'train_steps_per_second': 1.589, 'total_flos': 1037603113473600.0, 'train_loss': 0.47776405652364096, 'epoch': 5.0})

In [36]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.772     0.867     0.817       180
         yes      0.368     0.233     0.286        60

    accuracy                          0.708       240
   macro avg      0.570     0.550     0.551       240
weighted avg      0.671     0.708     0.684       240

[[156  24]
 [ 46  14]]


###distilbert multilingul

In [37]:
from transformers import AutoTokenizer

MODEL = f"distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [38]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset  = TextDataset(test_encodings,  y_test)

In [39]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [40]:
from transformers import DistilBertForSequenceClassification

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [41]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,0.719370
20,0.708422
30,0.657642
40,0.623095
50,0.650547
60,0.576116
70,0.641763
80,0.582792
90,0.599728
100,0.572215


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1200, training_loss=0.4483947302897771, metrics={'train_runtime': 332.9112, 'train_samples_per_second': 28.837, 'train_steps_per_second': 3.605, 'total_flos': 347784875958912.0, 'train_loss': 0.4483947302897771, 'epoch': 5.0})

In [42]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.802     0.856     0.828       180
         yes      0.458     0.367     0.407        60

    accuracy                          0.733       240
   macro avg      0.630     0.611     0.618       240
weighted avg      0.716     0.733     0.723       240

[[154  26]
 [ 38  22]]


#Experiment 2.C
comment + title + description as input

In [43]:
df.columns

Index(['StereoQueerEval_id', 'yt_title', 'yt_description', 'yt_comment',
       'stereotype', 'hate_speech', 'target'],
      dtype='object')

In [44]:
X = df.drop(['StereoQueerEval_id',
       'stereotype', 'hate_speech', 'target'], axis = 1)
y = df['stereotype'].map({'no': 0, 'yes': 1})

In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 45)
X_val, X_test, y_val, y_test = train_test_split(X, y, test_size = 0.5, random_state = 45)

In [46]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    desc  = d['yt_description'].fillna("").str[:300]        # trim long descriptions
    ctx = (title + " . " + desc)
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [47]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###cardiffnlp/twitter-roberta-base-hate-latest

In [48]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [49]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [50]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [51]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [52]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.306168
20,0.660849
30,0.705878
40,0.622166
50,0.571294
60,0.571763
70,0.651905
80,0.606062
90,0.630656
100,0.556363


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.5196122745672862, metrics={'train_runtime': 495.8524, 'train_samples_per_second': 19.361, 'train_steps_per_second': 1.21, 'total_flos': 1248519513226560.0, 'train_loss': 0.5196122745672862, 'epoch': 5.0})

In [53]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.840     0.872     0.855       865
         yes      0.632     0.570     0.600       335

    accuracy                          0.787      1200
   macro avg      0.736     0.721     0.728      1200
weighted avg      0.782     0.787     0.784      1200

[[754 111]
 [144 191]]


###Distilber-multilingual

In [54]:
from transformers import AutoTokenizer

MODEL = f"distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [55]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [56]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [57]:
from transformers import DistilBertForSequenceClassification

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [58]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,0.720044
20,0.702121
30,0.657767
40,0.605505
50,0.664764
60,0.586354
70,0.658008
80,0.584654
90,0.582439
100,0.568483


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1200, training_loss=0.48937389155228933, metrics={'train_runtime': 529.6489, 'train_samples_per_second': 18.125, 'train_steps_per_second': 2.266, 'total_flos': 508981141698432.0, 'train_loss': 0.48937389155228933, 'epoch': 5.0})

In [59]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.832     0.895     0.862       865
         yes      0.663     0.534     0.592       335

    accuracy                          0.794      1200
   macro avg      0.748     0.715     0.727      1200
weighted avg      0.785     0.794     0.787      1200

[[774  91]
 [156 179]]
